# Assignment 1: PID lab

Assignment 1 asks for two things: a complete PID controller in
`submission/controller.py`, and tuned gains in `submission/agent.yaml`. This
notebook is where you write and try both, first on a toy vehicle and then on the
public routes in MetaDrive. The gates and grading read only `submission/`, never
this notebook.

To solve the lab:

1. Work through Tasks 1 to 5. Each one is a `TODO` in **Your controller** and
   ends with the gate check that grades it. The **Checklist** shows all six
   implementation checks at once.
2. Tune the gains in **Your gains** on the public routes (Task 6).
3. Copy the controller and the gains into `submission/`, as the last section
   describes, and run its check.

After every change, rerun the cell you changed and every cell below it (in
JupyterLab, **Run > Run Selected Cell and All Below**).

Start the notebook from the repository root:

```bash
uv run --extra notebooks jupyter lab course/assignment-1/pid-lab.ipynb
```

In [ ]:
import logging
from dataclasses import replace

import matplotlib.pyplot as plt

from metadrive_starter.config import load_config
from metadrive_starter.gates import drive_route, load_gate_plan
from metadrive_starter.implementation_checks import (
    ANTI_WINDUP,
    LOAD_REJECTION,
    RESET,
    SPEED_DERIVATIVE,
    SPEED_INTEGRAL,
    STEERING_TERMS,
    LongitudinalModel,
    drive_speed,
    run_implementation_checks,
)
from metadrive_starter.submission import (
    PROJECT_ROOT,
    apply_submission_overlay,
    load_controller,
)
from metadrive_starter.types import ControlTick

SUBMISSION = PROJECT_ROOT / "submission"
PUBLIC_ROUTES = load_gate_plan(PROJECT_ROOT / "configs" / "gates-assignment-1.yaml")
FLAT = LongitudinalModel()
HILL = LongitudinalModel(load_mps2=0.5)
TARGET_MPS = 10.0

# MetaDrive logs every route it starts; keep only its warnings and errors.
logging.getLogger("MetaDrive").addFilter(lambda record: record.levelno >= logging.WARNING)


def submitted_config():
    """The configuration your submission runs with: the instructor configuration
    with submission/agent.yaml applied, read afresh."""
    return apply_submission_overlay(
        load_config(PROJECT_ROOT / "configs" / "default.yaml"), SUBMISSION
    )


def plot_speed(traces, title):
    """Speed against time for each trace, with the target as a dashed line."""
    figure, axis = plt.subplots(figsize=(8, 3.5))
    for label, trace in traces.items():
        axis.plot(trace.time_s, trace.speed_mps, label=label)
    first = next(iter(traces.values()))
    axis.plot(first.time_s, first.target_mps, "k--", linewidth=1, label="target")
    axis.set_title(title)
    axis.set_xlabel("time (s)")
    axis.set_ylabel("speed (m/s)")
    axis.legend()
    plt.show()


def check(*names):
    """Run the gates' implementation checks on the controller DRIVE chooses, and
    print how each named one went."""
    results = {result.check: result for result in run_implementation_checks(build)}
    for name in names:
        result = results[name]
        print(f"{'PASS   ' if result.passed else 'NOT YET'}  {result.check}: {result.detail}")

## Your controller

This cell starts as a copy of the shipped `submission/controller.py`, below its
docstring. Each `TODO` in it is a task below:

| `TODO` | Task |
| --- | --- |
| `Controller.__init__`: remove the weak speed gain | 1 |
| `PID.__init__` and `PID.update`: the integral term | 2 |
| `PID.update`: the derivative term | 3 |
| `PID.update`: anti-windup | 4 |
| `PID.reset`: clear the history | 5 |

The speed, heading, and lateral loops are all a `PID`, so completing the class
completes all three.

In [ ]:
from __future__ import annotations

from dataclasses import replace

from metadrive_starter.config import ControllerSettings, PIDSettings
from metadrive_starter.types import Action, ControlTick


class PID:
    """One control loop. Only the proportional term is implemented."""

    def __init__(self, gains: PIDSettings) -> None:
        self.gains = gains
        # TODO: keep the history the integral and derivative terms need.

    def reset(self) -> None:
        """Forget loop history. A proportional-only loop keeps none."""
        # TODO: clear the history that __init__ keeps.

    def update(self, setpoint: float, measurement: float, dt: float) -> float:
        error = setpoint - measurement
        # TODO: add the integral term (ki) and the derivative term (kd). Then add
        # anti-windup: while the output is saturated in the direction the error
        # pushes, stop the integral growing, unless self.gains.anti_windup is False.
        output = self.gains.kp * error
        return max(self.gains.output_min, min(self.gains.output_max, output))


class Controller:
    """Throttle and brake from a speed loop; steering from heading and lateral loops."""

    def __init__(self, settings: ControllerSettings) -> None:
        # Deliberately weak, and it overrides the configured speed_pid.kp: a
        # proportional-only speed loop with this gain settles well below target.
        # TODO: remove this override, so the speed loop uses settings.speed_pid.
        self.speed_loop = PID(replace(settings.speed_pid, kp=0.03))
        self.heading_loop = PID(settings.steering_pid)
        self.lateral_loop = PID(settings.lateral_pid)

    def update(self, tick: ControlTick) -> Action:
        throttle_brake = self.speed_loop.update(
            tick.target_speed_mps,
            tick.speed_mps,
            tick.dt_s,
        )
        # Each steering loop drives its error to zero: the setpoint is 0 and the
        # measured deviation is -error, so a positive error steers left.
        steering = self.heading_loop.update(0.0, -tick.heading_error_rad, tick.dt_s)
        steering += self.lateral_loop.update(0.0, -tick.lateral_error_m, tick.dt_s)
        return (max(-1.0, min(1.0, steering)), throttle_brake)

    def reset_speed_control(self) -> None:
        self.speed_loop.reset()

## Your gains

The gains Task 6 drives the routes with. They start as your submission's:
`configs/pid.yaml` with `submission/agent.yaml` applied. Tasks 1 to 5 use
demonstration gains of their own, so they test your code whatever you set here.

`DRIVE` chooses the controller every task drives: `"notebook"`, the one above, or
`"submission"`, your `submission/controller.py`, loaded exactly as the gates load
it.

In [ ]:
# TODO (Task 6): tune these gains. Change a value with replace(), for example
#     SPEED = replace(SPEED, kp=0.3, ki=0.05)
# then rerun this cell and every cell below it.
SPEED = submitted_config().controller.speed_pid
STEERING = submitted_config().controller.steering_pid
LATERAL = submitted_config().controller.lateral_pid

DRIVE = "notebook"

settings = replace(
    submitted_config().controller,
    speed_pid=SPEED,
    steering_pid=STEERING,
    lateral_pid=LATERAL,
)


def build(settings):
    """A controller to drive, chosen by DRIVE."""
    factory = Controller if DRIVE == "notebook" else load_controller(SUBMISSION)
    return factory(settings)


def demo(**speed_gains):
    """A controller whose speed loop has these demonstration gains, and zero for
    any gain not given."""
    gains = {"kp": 0.0, "ki": 0.0, "kd": 0.0, "anti_windup": True, **speed_gains}
    return build(replace(settings, speed_pid=replace(settings.speed_pid, **gains)))


print(f"driving the {DRIVE} controller")
for name, gains in (("speed", SPEED), ("steering", STEERING), ("lateral", LATERAL)):
    print(f"{name:>8}: kp = {gains.kp:g}, ki = {gains.ki:g}, kd = {gains.kd:g}")

## Checklist

The six implementation checks the gates run, on the controller `DRIVE` chooses.
They make up half of the score, and each builds your controller with gains of its
own. The shipped controller passes none of them. Rerun this cell after each task.

In [ ]:
results = run_implementation_checks(build)
for result in results:
    print(f"{'PASS   ' if result.passed else 'NOT YET'}  {result.check}")
passed = sum(result.passed for result in results)
print(f"\n{passed} of {len(results)} implementation checks pass.")

## The toy vehicle

Tasks 1 to 5 drive a toy vehicle instead of MetaDrive, so each experiment takes a
moment. `LongitudinalModel` is a straight road and a car whose acceleration is
proportional to throttle, 2.76 m/s² at full throttle, or to brake, 11.2 m/s² at
full brake, both measured in MetaDrive. Like MetaDrive's car, it needs no throttle
to hold its speed on a flat road. `HILL` adds a steady 0.5 m/s² deceleration, about
a 5% grade, which MetaDrive's flat maps never have.

`drive_speed(controller, target, duration_s=...)` calls `controller.update(tick)`
every 0.1 s, as the simulator does, and returns the speed and output on every tick.

## Task 1: remove the weak speed gain

`TODO` in `Controller.__init__`. The shipped controller replaces the configured
speed gain with `kp = 0.03`, so it ignores whatever gain it is given: here, a
standing start with `kp = 0.3`. Remove the override, so the speed loop uses
`settings.speed_pid` as configured. With it gone, the car is within 0.1 m/s of
the target in under 7 s; with it, the car is still almost 2 m/s short after 20 s.

Every later task needs this one: until it is done, the speed loop ignores their
demonstration gains too.

In [ ]:
plot_speed(
    {"kp = 0.3": drive_speed(demo(kp=0.3), TARGET_MPS, duration_s=20.0, model=FLAT)},
    "Standing start on a flat road",
)
throttle = demo(kp=0.3).update(ControlTick(TARGET_MPS, TARGET_MPS - 1.0, 0.0, 0.0, 0.1))[1]
if abs(throttle - 0.3) < 1e-9:
    print("PASS     the speed loop uses the configured kp: a 1 m/s error gives throttle 0.30")
else:
    print(
        f"NOT YET  with kp = 0.3, a 1 m/s error gives throttle {throttle:.2f}, not 0.30: "
        "the speed loop does not use the configured kp"
    )

## Task 2: the integral term

`TODO` in `PID.__init__` and `PID.update`. On a hill the car needs some throttle
just to hold its speed, and a proportional-only loop gives throttle only when
there is an error, so it settles below the target: here 0.5 / (2.76 × 0.3) ≈ 0.6
m/s below. The integral term adds `ki` × the error summed over time, which keeps
growing until the speed is back on target.

With a working integral term, the `ki = 0.1` curve climbs back to the target.
Until then, the two curves are the same.

In [ ]:
plot_speed(
    {
        "kp = 0.3": drive_speed(
            demo(kp=0.3), TARGET_MPS, duration_s=40.0, initial_speed_mps=TARGET_MPS, model=HILL
        ),
        "kp = 0.3, ki = 0.1": drive_speed(
            demo(kp=0.3, ki=0.1),
            TARGET_MPS,
            duration_s=40.0,
            initial_speed_mps=TARGET_MPS,
            model=HILL,
        ),
    },
    "Cruising onto a 5% grade",
)
check(SPEED_INTEGRAL, LOAD_REJECTION)

## Task 3: the derivative term

`TODO` in `PID.update`. The derivative term adds `kd` × how fast the error
changes, per second. Below, the car speeds up by 1 m/s every second while the
target holds, so the error falls by 1 m/s every second, and with `kd = 0.1` the
derivative term should add -0.100 on every tick after the first. The cell
measures it as the gates do: the difference between the controller with
`kd = 0.1` and the same controller without it. Until the term is implemented, the
difference is 0.000.

This car has no lag for the derivative term to damp, so on the routes it buys
little. The gates check that it is there and correct, in the speed loop and in
the steering loops.

In [ ]:
with_kd, without_kd = demo(kd=0.1), demo()
added = []
for tick in range(10):
    speeding_up = ControlTick(TARGET_MPS, 5.0 + 0.1 * tick, 0.0, 0.0, 0.1)
    added.append(with_kd.update(speeding_up)[1] - without_kd.update(speeding_up)[1])
print("kd = 0.1 adds, on ticks 2 to 10:", " ".join(f"{value:+.3f}" for value in added[1:]))
check(SPEED_DERIVATIVE, STEERING_TERMS)

## Task 4: anti-windup

`TODO` in `PID.update`. A standing start up the hill keeps the output saturated
at full throttle for seconds. Without anti-windup the integral keeps growing all
that time, and it has to unwind before the throttle comes off, so the car
overshoots its target. Stop the integral growing while the output is saturated in
the direction the error pushes, unless `anti_windup` is false.

With working anti-windup, the `anti_windup = True` curve overshoots far less than
the other. Until then, the two curves are the same.

In [ ]:
windup = {
    f"anti_windup = {anti_windup}": drive_speed(
        demo(kp=0.3, ki=0.1, anti_windup=anti_windup), TARGET_MPS, duration_s=40.0, model=HILL
    )
    for anti_windup in (True, False)
}
plot_speed(windup, "Standing start up a 5% grade, kp = 0.3, ki = 0.1")
for label, trace in windup.items():
    peak = max(trace.speed_mps)
    if peak > TARGET_MPS:
        print(f"{label}: overshoots the target by {peak - TARGET_MPS:.2f} m/s")
    else:
        print(f"{label}: never reaches the target, {TARGET_MPS - peak:.2f} m/s short")
check(ANTI_WINDUP)

## Task 5: resetting speed history

`TODO` in `PID.reset`. On every tick on which the safety floor overrides your
throttle or brake, the simulator calls `reset_speed_control()`, which calls the
speed loop's `reset()`, so history built up while the floor held the car cannot
push it once the floor lets go. Here a slower car holds you at 8 m/s for 10 s while your
target is 10 m/s, and then the road clears.

With a working `reset()`, the reset car accelerates gently back to its target and
the other surges past it. Until then, the two curves are the same.

In [ ]:
def released_after_following(reset_each_tick):
    controller = demo(kp=0.3, ki=0.1)
    held = ControlTick(TARGET_MPS, 8.0, 0.0, 0.0, 0.1)
    for _ in range(100):
        controller.update(held)
        if reset_each_tick:
            controller.reset_speed_control()
    return drive_speed(controller, TARGET_MPS, duration_s=15.0, initial_speed_mps=8.0, model=FLAT)


plot_speed(
    {
        "reset on every held tick": released_after_following(True),
        "never reset": released_after_following(False),
    },
    "Released after following a slower car for 10 s",
)
check(RESET)

## Task 6: tune on the routes

`TODO` in **Your gains**. With the PID complete, the gains decide the other half
of the score: how your car drives the routes. The cell below drives the five
public routes in MetaDrive with the controller `DRIVE` chooses and the gains in
**Your gains**, and prints what the gate report prints for each route. It takes
about a minute.

A route earns full credit for arriving within its par steps, with a speed error
of at most 0.1 m/s and a lateral error of at most 0.02 m; "The score" in the brief
gives the details. The starting gains are deliberately untuned. Change them, rerun
**Your gains** and this cell, and compare. A route that leaves the road earns
nothing, so a stiffer loop is not always better.

In [ ]:
config = replace(submitted_config(), controller=settings)
credits = []
for scenario in PUBLIC_ROUTES.scenarios:
    route = drive_route(build, scenario, config=config)
    credits.append(route.credit)
    if route.problem is not None:
        print(f"{100 * route.credit:5.1f}  {route.scenario_id}: {route.problem}")
    else:
        print(
            f"{100 * route.credit:5.1f}  {route.scenario_id}: {route.steps} steps against "
            f"par {route.par_steps}, speed error {route.speed_error_mps:.2f} m/s, "
            f"lateral error {route.lateral_error_m:.2f} m"
        )
print(f"\ndriving credit {100 * sum(credits) / len(credits):.1f} of 100")

## Copy your work into submission/

The gates and grading read only `submission/controller.py` and
`submission/agent.yaml`, so copy this notebook's work into them:

1. In `submission/controller.py`, replace everything below the docstring with the
   code in **Your controller**.
2. Put the gains you changed in **Your gains** into `submission/agent.yaml`,
   nested as in `configs/pid.yaml`, for example:

```yaml
controller:
  speed_pid:
    kp: 0.5
    ki: 0.02
```

Then run the check below. It compares your submission with this notebook, gains
and behaviour, on the speed loop and the steering loops.

In [ ]:
submitted_settings = submitted_config().controller
loops = ("speed_pid", "steering_pid", "lateral_pid")
different_gains = [
    loop for loop in loops if getattr(submitted_settings, loop) != getattr(settings, loop)
]


def actions(controller):
    """The controller's output over a run that moves every error and resets once."""
    out = []
    for tick in range(200):
        if tick == 100:
            controller.reset_speed_control()
        out.append(
            controller.update(
                ControlTick(TARGET_MPS, 6.0 + 0.04 * tick, 0.05 - 0.0005 * tick, 0.3 - 0.003 * tick, 0.1)
            )
        )
    return out


same_code = actions(load_controller(SUBMISSION)(settings)) == actions(Controller(settings))
copied = same_code and not different_gains
if different_gains:
    print(
        f"The gains differ in {', '.join(different_gains)}: put the gains from "
        "Your gains into submission/agent.yaml, then rerun this cell."
    )
if not same_code:
    print(
        "submission/controller.py drives differently from Your controller: copy the "
        "code below its docstring again, then rerun this cell."
    )
if copied:
    passing = sum(result.passed for result in run_implementation_checks(load_controller(SUBMISSION)))
    if passing < 6:
        print(
            f"submission/ matches this notebook, but only {passing} of 6 implementation "
            "checks pass: finish the tasks above, then copy again."
        )
    else:
        print(
            "submission/ matches this notebook, and all six implementation checks pass. "
            "See your score with: uv run metadrive-starter gates --submission submission"
        )